# Introduction to Linear Regression: Predicting Oil Volume
In this exercise, we will use historical production data from the Volve field to build a linear regression model. 
Our goal is to predict the volume of oil produced (`BORE_OIL_VOL`) based on downhole pressure and choke size.

In [ ]:
# Step 1: Import the Pandas package and Matplotlib for plotting
import pandas as pd
import matplotlib.pyplot as plt

# Ensure plots show up inline in the notebook
%matplotlib inline

## Loading and Understanding the Data
We start by importing the data into a dataframe called `df` and previewing it.

In [ ]:
file_path = "Volve_Production_Data.csv" 
df = pd.read_csv(file_path)

# Preview the first 5 rows
display(df.head())

Let's get a concise summary of the structure of the data and its statistics.

In [ ]:
# Get structural summary
print(df.info())

# Get statistical summary (mean, min, max, standard deviation)
display(df.describe())

## Essential Data Cleaning
Unlike perfectly curated practice datasets, real industrial data has missing values. 
A Linear Regression model cannot handle blank spaces (`NaN`), so we must clean the data first.
1. We will drop any rows where our target (Oil Volume) is completely missing.
2. We will fill any missing sensor readings with the average (mean) of that sensor.

In [ ]:
# Drop rows where the target variable is missing
df = df.dropna(subset=['BORE_OIL_VOL'])

# Fill remaining missing values with the mean of their respective columns
df = df.fillna(df.mean(numeric_only=True))

print("Data cleaning complete. No missing values remain.")

## Visualizing Linear Relationships
Linear regression models assume there is a relationship between the predictors and the response. 
Let's see if this assumption holds true for our dataset using scatter plots.

In [ ]:
# Plot 1: Downhole Pressure vs Oil Volume
plt.scatter(df['AVG_DOWNHOLE_PRESSURE'], df['BORE_OIL_VOL'], alpha=0.5)
plt.title("Relationship between Downhole Pressure and Oil Volume")
plt.xlabel("Average Downhole Pressure")
plt.ylabel("Oil Volume Produced")
plt.show()

# Plot 2: Choke Size vs Oil Volume
plt.scatter(df['AVG_CHOKE_SIZE_P'], df['BORE_OIL_VOL'], alpha=0.5, color='green')
plt.title("Relationship between Choke Size and Oil Volume")
plt.xlabel("Average Choke Size")
plt.ylabel("Oil Volume Produced")
plt.show()

## Splitting the Data
Before we build our machine learning model, we need to split the data into training and test sets.
First, we separate the dependent variable (Y) from the independent variables (X).

In [ ]:
# Separate the dependent variable (Target)
Y = df['BORE_OIL_VOL']

# Separate the independent variables (Features)
X = df[['AVG_DOWNHOLE_PRESSURE', 'AVG_CHOKE_SIZE_P']]

Next, we import the `train_test_split` function from the sklearn package and split our X and Y dataframes. 
We will use 70% of the data to train the model, and 30% to test it.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

print(f"Training data size: {X_train.shape[0]} rows")
print(f"Testing data size: {X_test.shape[0]} rows")

## Building the Linear Regression Model
To build a linear regression model in Python, we import the `LinearRegression` class. 
We then call the `fit` method and pass our training data to it.

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

# Fit the model to the training data
model.fit(X_train, Y_train)

## Interpreting the Model
The objective of linear regression is to estimate the intercept and slope values that best fit the data.
Let's look at the estimated intercept and the coefficients (slopes) for our features.

In [ ]:
# Get the intercept
print(f"Intercept: {model.intercept_:.2f}")

# Get the coefficients
print(f"Coefficient for Downhole Pressure: {model.coef_[0]:.2f}")
print(f"Coefficient for Choke Size: {model.coef_[1]:.2f}")

*Note on interpretation:* If the coefficient for choke size is positive, it means that as the choke size increases by 1 unit, the expected oil volume increases by that coefficient amount, assuming pressure stays exactly the same.

## Evaluating the Model
One way to evaluate a model is by calculating the coefficient of determination, or R-squared. 
The closer this metric is to 1.0, the better the model explains the variance in the data.

In [ ]:
# Calculate R-squared using the test data
r_squared = model.score(X_test, Y_test)
print(f"R-squared Score: {r_squared:.4f}")

Another way to evaluate the model is to see how accurate it is on average. 
We can do this by generating predictions and comparing them to the actual test values using Mean Absolute Error.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Get the model's predicted values for the test data
Y_pred = model.predict(X_test)

# Calculate the Mean Absolute Error
mae = mean_absolute_error(Y_test, Y_pred)

print(f"Mean Absolute Error: {mae:.2f}")
print(f"This means our predictions are off by an average of +/- {mae:.2f} units of oil volume.")